In [3]:
import glob
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [4]:
# Load data
folder = "../Preprocessing-FeatureExtraction/cleaned-data/"
csv_files = glob.glob(os.path.join(folder, "*.csv"))
dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    df['state'] = os.path.splitext(os.path.basename(file))[0] 
    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)
df = df.dropna()
df['sentiment'] = df['stars'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))
print(f"Loaded {len(df)} reviews from {len(dfs)} states.")
sample = df.sample(1000000)
sample.head()

Loaded 5222860 reviews from 20 states.


,stars,text,review_length,num_exclamations,num_caps_words,clean_text,state,sentiment
2550245,5.0,Going to keep with my 5 star because this plac...,66,0,1,going keep star place bomb inside clean staff ...,FL,2
2239439,5.0,"Excellent service, high quality food. Dined w...",25,1,0,excellent service high quality food dined part...,FL,2
3128508,5.0,This place is awesome and a great value. You g...,44,0,0,place awesome great value get generous portion...,PA,2
1925960,2.0,We stopped in for a quick bite on St. Paddy's ...,143,0,7,stopped quick bite st paddys day half full sur...,FL,0
1641648,5.0,Staple of Santa Barbara - really isn't a dish ...,16,1,0,staple santa barbara really isnt dish order wo...,CA,2


In [5]:
# Encode text with pretrained embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2")
X_embedded = embedder.encode(sample['clean_text'].tolist(), batch_size=128, show_progress_bar=True)
# sample["embedding"] = np.array(X_embedded)
# sample.head()

Batches:   0%|          | 0/7813 [00:00<?, ?it/s]

ValueError: Expected a 1D array, got an array with shape (1000000, 384)

In [8]:
folder_name = 'extracted-features'
if not os.path.exists(folder_name):
    os.makedirs(folder_name)

# Save embeddings as .npy
np.save(os.path.join(folder_name, "sentencetransformer_embeddings.npy"), X_embedded)

# Save metadata without embeddings
sample.to_csv(os.path.join(folder_name, "sentencetransformer_meta.csv"), index=False)

print("Embeddings and metadata saved successfully!")

Embeddings and metadata saved successfully!
